# Volatility Regime Switch Strategy Demonstration

This notebook recreates `qc_projects/main_volatility_regime.py`, which:
- Computes 20-day annualized SPY volatility
- Classifies market into low / medium / high volatility regimes
- Rotates between aggressive and defensive asset allocations on regime changes

## Step 1: Imports and Parameters

We keep the same thresholds used in QC: low < 15%, high > 25%.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

symbols = {
    'spy': 'SPY',
    'qqq': 'QQQ',
    'arkk': 'ARKK',
    'tlt': 'TLT',
    'gld': 'GLD',
}
start_date = '2020-01-01'
end_date = '2022-01-01'
vol_window = 20
high_vol_threshold = 25.0
low_vol_threshold = 15.0
warmup_period = vol_window + 5

print(f"Window: {start_date} → {end_date}")
print(f"Vol window: {vol_window} | Thresholds: low<{low_vol_threshold}, high>{high_vol_threshold}")
print(f"Symbols: {symbols}")

## Step 2: Download and Align Asset Data

We pull daily closes for SPY/QQQ/ARKK/TLT/GLD and align to common dates.

In [ ]:
close_map = {}
for key, sym in symbols.items():
    df = yf.download(
        sym,
        start=start_date,
        end=end_date,
        interval='1d',
        progress=False,
        multi_level_index=False,
        auto_adjust=False,
    )
    if df.empty:
        raise RuntimeError(f'No data for {sym}')
    close_map[key] = df['Close']

prices = pd.DataFrame(close_map).dropna().copy()
prices.index = pd.to_datetime(prices.index).tz_localize(None)
prices.index.name = 'date'

print(f'Aligned rows: {len(prices):,}')
prices.head()

## Step 3: Compute Rolling Volatility and Regimes

We annualize the rolling standard deviation of SPY daily returns.

In [ ]:
features = prices.copy()
features['spy_ret'] = features['spy'].pct_change()
features['market_vol'] = features['spy_ret'].rolling(vol_window).std(ddof=0) * np.sqrt(252) * 100

for asset in ['qqq', 'arkk', 'tlt', 'gld']:
    features[f'ret_{asset}'] = features[asset].pct_change().fillna(0.0)

features = features.dropna(subset=['market_vol']).copy()
if len(features) > warmup_period:
    features = features.iloc[warmup_period:].copy()

features[['spy', 'market_vol', 'ret_qqq', 'ret_arkk', 'ret_tlt', 'ret_gld']].head()

## Step 4: Replay Regime Allocation Logic

Regime allocations from QC:
- High vol: 50% TLT, 40% GLD, 10% cash
- Low vol: 60% QQQ, 30% ARKK, 10% cash
- Medium vol: 30% QQQ, 30% TLT, 20% GLD, 20% cash

In [ ]:
current_regime = None
weights = {'qqq': 0.0, 'arkk': 0.0, 'tlt': 0.0, 'gld': 0.0, 'cash': 1.0}

regime_events = []
weights_history = []
portfolio_returns = []

for ts, row in features.iterrows():
    vol = float(row['market_vol'])

    if vol > high_vol_threshold:
        new_regime = 'high'
        target_weights = {'qqq': 0.0, 'arkk': 0.0, 'tlt': 0.5, 'gld': 0.4, 'cash': 0.1}
    elif vol < low_vol_threshold:
        new_regime = 'low'
        target_weights = {'qqq': 0.6, 'arkk': 0.3, 'tlt': 0.0, 'gld': 0.0, 'cash': 0.1}
    else:
        new_regime = 'medium'
        target_weights = {'qqq': 0.3, 'arkk': 0.0, 'tlt': 0.3, 'gld': 0.2, 'cash': 0.2}

    if new_regime != current_regime:
        regime_events.append({
            'timestamp': ts,
            'regime': new_regime,
            'market_vol': round(vol, 3),
            **target_weights,
        })
        current_regime = new_regime
        weights = target_weights

    day_ret = (
        weights['qqq'] * float(row['ret_qqq'])
        + weights['arkk'] * float(row['ret_arkk'])
        + weights['tlt'] * float(row['ret_tlt'])
        + weights['gld'] * float(row['ret_gld'])
    )
    portfolio_returns.append(day_ret)

    weights_history.append({
        'timestamp': ts,
        'regime': current_regime,
        'market_vol': round(vol, 3),
        **weights,
        'daily_return_pct': round(day_ret * 100, 4),
    })

events_df = pd.DataFrame(regime_events)
weights_df = pd.DataFrame(weights_history)
print(f'Regime changes: {len(events_df)}')
events_df.head() if not events_df.empty else 'No regime changes'

## Step 5: Performance Metrics

Compute portfolio-level returns and risk statistics from the regime-switch allocations.

In [ ]:
rets = np.array(portfolio_returns, dtype=float)
if rets.size == 0:
    summary = {
        'regime_changes': 0,
        'total_return_pct': 0.0,
        'annualized_return_pct': 0.0,
        'annualized_vol_pct': 0.0,
        'sharpe': 0.0,
        'max_drawdown_pct': 0.0,
    }
else:
    equity = np.cumprod(1.0 + rets)
    total_return = equity[-1] - 1.0
    years = max(len(rets) / 252.0, 1e-9)
    annualized_return = equity[-1] ** (1.0 / years) - 1.0
    annualized_vol = np.std(rets, ddof=0) * np.sqrt(252)
    sharpe = annualized_return / annualized_vol if annualized_vol > 0 else 0.0
    peak = np.maximum.accumulate(equity)
    drawdown = equity / peak - 1.0
    max_dd = drawdown.min() if drawdown.size else 0.0

    summary = {
        'regime_changes': int(len(events_df)),
        'total_return_pct': round(float(total_return) * 100, 3),
        'annualized_return_pct': round(float(annualized_return) * 100, 3),
        'annualized_vol_pct': round(float(annualized_vol) * 100, 3),
        'sharpe': round(float(sharpe), 3),
        'max_drawdown_pct': round(float(max_dd) * 100, 3),
    }

summary

## Step 6: Visualize Volatility Regimes and Allocation Timeline

We plot market volatility with regime thresholds and show allocation drift over time.

In [ ]:
graphs_dir = Path('graphs')
graphs_dir.mkdir(exist_ok=True)

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.35, 0.35, 0.30],
    subplot_titles=('SPY Market Volatility', 'Regime Portfolio Weights', 'Daily Regime Portfolio Return (%)'),
)

fig.add_trace(go.Scatter(x=features.index, y=features['market_vol'], name='Market Vol (%)', line=dict(color='#1f77b4')), row=1, col=1)
fig.add_hline(y=high_vol_threshold, line=dict(color='#d62728', dash='dash'), row=1, col=1)
fig.add_hline(y=low_vol_threshold, line=dict(color='#2ca02c', dash='dash'), row=1, col=1)

if not weights_df.empty:
    fig.add_trace(go.Scatter(x=weights_df['timestamp'], y=weights_df['qqq'], name='QQQ weight', line=dict(color='#9467bd')), row=2, col=1)
    fig.add_trace(go.Scatter(x=weights_df['timestamp'], y=weights_df['arkk'], name='ARKK weight', line=dict(color='#ff7f0e')), row=2, col=1)
    fig.add_trace(go.Scatter(x=weights_df['timestamp'], y=weights_df['tlt'], name='TLT weight', line=dict(color='#2ca02c')), row=2, col=1)
    fig.add_trace(go.Scatter(x=weights_df['timestamp'], y=weights_df['gld'], name='GLD weight', line=dict(color='#bcbd22')), row=2, col=1)
    fig.add_trace(go.Scatter(x=weights_df['timestamp'], y=weights_df['daily_return_pct'], name='Daily return %', line=dict(color='#7f7f7f')), row=3, col=1)

if not events_df.empty:
    fig.add_trace(
        go.Scatter(
            x=events_df['timestamp'],
            y=events_df['market_vol'],
            mode='markers',
            marker=dict(symbol='diamond', size=9, color='#d62728'),
            name='Regime changes',
        ),
        row=1, col=1
    )

fig.update_layout(height=900, title='Volatility Regime Switch Strategy Diagnostics')
output_path = graphs_dir / 'volatility_regime_signals.html'
fig.write_html(output_path)
fig.show()
print(f'Saved interactive chart to {output_path}')

## Step 7: Regime Event Log and Recommendation

Review the latest regime transitions and print a quick recommendation.

In [ ]:
if not events_df.empty:
    display(events_df.tail(12))

if summary['regime_changes'] == 0:
    recommendation = 'HOLD - insufficient volatility regime transitions in selected window.'
elif summary['sharpe'] > 0.7 and summary['max_drawdown_pct'] > -15:
    recommendation = 'BUY (strategy-ready) - attractive risk-adjusted profile.'
elif summary['total_return_pct'] <= 0:
    recommendation = 'AVOID/REFINE - adjust thresholds or asset mix.'
else:
    recommendation = 'NEUTRAL - mixed profile, continue tuning.'

print('Recommendation:', recommendation)

## Recap

- This demo mirrors your QC volatility-regime allocation state machine.
- Use it to tune `vol_window`, thresholds, and asset baskets before pushing to Lean.
- Saved chart: `graphs/volatility_regime_signals.html`.